In [9]:
# ============================================================
# CELL 8 — monitoring_pipeline.py
# Monthly model health check: PSI · KS · Feature Drift · Default rate
#
# Uses   →  master   (DataFrame from Cell 6)
#            model    (GBM from Cell 7)
#            features (feature list from Cell 7)
# Reads  →  config/scoring_cutoff.yaml  (PSI thresholds)
# Produces → monitoring_report   (dict)
#             drift_report        (DataFrame)
#             Saves PNG + JSON + CSV to models/
# ============================================================

import os
import json

from datetime import datetime

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_curve

import matplotlib
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────
OUT_DIR       = "models"
TARGET_COL    = "target_default"
PSI_BINS      = 10
PSI_THRESHOLD = 0.20
RUN_LABEL     = datetime.now().strftime("%Y%m")

# In a real monthly run, current_df would be the new month's scored data.
# Here we use master as both baseline and current (same data = PSI≈0 expected).
# Swap current_df for the live month's DataFrame when available.
BASE_PATH = "D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine"

try:
    assert isinstance(master, pd.DataFrame)
except (NameError, AssertionError):
    master = pd.read_csv(f"{BASE_PATH}/data/processed/model_dataset.csv")
    print(f"  master loaded from disk : {master.shape[0]:,} rows × {master.shape[1]} cols")

try:
    assert hasattr(model, "predict_proba")
except (NameError, AssertionError):
    model = joblib.load(f"{BASE_PATH}/models/gbm_model.pkl")
    print(f"  model loaded from disk  : {type(model).__name__}")

try:
    assert isinstance(features, list) and len(features) > 0
except (NameError, AssertionError):
    with open(f"{BASE_PATH}/models/feature_list.json") as _f:
        features = json.load(_f)
    print(f"  features loaded         : {len(features)} features")

baseline_df = master.copy()
current_df  = master.copy()

# ── INLINED: PSI helpers ──────────────────────────────────────

def _psi_status(v: float) -> str:
    if v < 0.10: return "STABLE"
    if v < 0.20: return "MONITOR"
    return "RETRAIN"


def _calculate_psi(expected: np.ndarray, actual: np.ndarray,
                   n_bins: int = PSI_BINS) -> float:
    """
    Population Stability Index.
    Measures how much a distribution has shifted since baseline.
      PSI < 0.10  → STABLE   (no action needed)
      PSI 0.10–0.20 → MONITOR (watch closely)
      PSI > 0.20  → RETRAIN  (model likely stale)
    """
    eps  = 1e-4
    bins = np.unique(np.percentile(expected, np.linspace(0, 100, n_bins + 1)))
    if len(bins) < 3:
        return 0.0
    ep = np.histogram(expected, bins=bins)[0]
    ap = np.histogram(actual,   bins=bins)[0]
    ep_pct = np.where(ep == 0, eps, ep / len(expected))
    ap_pct = np.where(ap == 0, eps, ap / len(actual))
    return float(np.sum((ap_pct - ep_pct) * np.log(ap_pct / ep_pct)))


def _psi_by_feature(baseline: pd.DataFrame, current: pd.DataFrame,
                    cols: list) -> pd.DataFrame:
    """PSI for every numeric feature — returns DataFrame sorted by PSI desc."""
    rows = []
    for col in cols:
        if col not in baseline.columns or col not in current.columns:
            continue
        try:
            b = baseline[col].dropna().values
            c = current[col].dropna().values
            if len(b) < 10 or len(c) < 10:
                continue
            v = _calculate_psi(b, c)
            rows.append({"feature": col, "psi": round(v, 4),
                         "status": _psi_status(v)})
        except Exception:
            pass
    return pd.DataFrame(rows).sort_values("psi", ascending=False).reset_index(drop=True)


# ── INLINED: KS helpers ───────────────────────────────────────

def _ks_status(v: float) -> str:
    if v >= 0.40: return "EXCELLENT"
    if v >= 0.30: return "GOOD"
    if v >= 0.20: return "ACCEPTABLE"
    return "POOR"


def _calculate_ks(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """KS = max(TPR − FPR) over all score thresholds."""
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    return float(np.max(tpr - fpr))


def _ks_decile_table(y_true: np.ndarray, y_pred: np.ndarray,
                     n_deciles: int = 10) -> pd.DataFrame:
    """
    Rank-order decile table from highest risk (decile 1) to lowest.
    Shows cumulative bad % and good % per decile — KS is the max gap.
    """
    df = pd.DataFrame({"y": y_true, "p": y_pred})
    df = df.sort_values("p", ascending=False).reset_index(drop=True)
    df["decile"] = pd.qcut(df["p"].rank(method="first"),
                           n_deciles, labels=False, duplicates="drop")

    total_bad  = max(df["y"].sum(), 1)
    total_good = max(len(df) - total_bad, 1)
    rows, cum_b, cum_g = [], 0, 0

    for dec in range(n_deciles - 1, -1, -1):
        grp = df[df["decile"] == dec]
        b   = grp["y"].sum()
        g   = len(grp) - b
        cum_b += b
        cum_g += g
        rows.append({
            "decile":       n_deciles - dec,
            "n":            len(grp),
            "bads":         int(b),
            "goods":        int(g),
            "bad_rate_pct": round(b / max(len(grp), 1) * 100, 2),
            "cum_bad_pct":  round(cum_b / total_bad  * 100, 2),
            "cum_good_pct": round(cum_g / total_good * 100, 2),
            "ks_pct":       round(abs(cum_b / total_bad - cum_g / total_good) * 100, 2),
        })
    return pd.DataFrame(rows)


# ── INLINED: Drift helpers ────────────────────────────────────

def _detect_numeric_drift(b_series: pd.Series,
                           c_series: pd.Series) -> dict:
    """PSI + two-sample KS test for a numeric feature."""
    b, c = b_series.dropna().values, c_series.dropna().values
    if len(b) < 5 or len(c) < 5:
        return {"psi": 0.0, "status": "INSUFFICIENT_DATA",
                "ks_stat": None, "ks_pval": None, "alert": False}
    psi_val           = _calculate_psi(b, c)
    ks_stat, ks_pval  = stats.ks_2samp(b, c)
    return {
        "psi":     round(psi_val, 4),
        "status":  _psi_status(psi_val),
        "ks_stat": round(ks_stat, 4),
        "ks_pval": round(ks_pval, 4),
        "alert":   psi_val > PSI_THRESHOLD or ks_pval < 0.05,
    }


def _run_drift_report(baseline: pd.DataFrame,
                       current: pd.DataFrame,
                       feature_cols: list) -> pd.DataFrame:
    """Drift report for every numeric feature in feature_cols."""
    results = []
    for col in feature_cols:
        if col not in baseline.columns or col not in current.columns:
            continue
        try:
            if pd.api.types.is_numeric_dtype(baseline[col]):
                d = _detect_numeric_drift(baseline[col], current[col])
                d["feature"] = col
                d["type"]    = "numeric"
                results.append(d)
        except Exception:
            pass
    if not results:
        return pd.DataFrame()
    return (pd.DataFrame(results)
            .sort_values("psi", ascending=False)
            .reset_index(drop=True))


# ── INLINED: Chart generator ──────────────────────────────────

def _save_monitoring_chart(p_base, p_curr, decile_tbl, out_dir, label):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Monitoring Report — {label}", fontsize=14, fontweight="bold")

    # Panel 1: Score distribution overlay
    axes[0].hist(p_base, bins=30, alpha=0.5, color="steelblue",
                 label="Baseline", density=True)
    axes[0].hist(p_curr, bins=30, alpha=0.5, color="coral",
                 label="Current",  density=True)
    axes[0].set_xlabel("Predicted PD")
    axes[0].set_ylabel("Density")
    axes[0].set_title("PD Distribution — Baseline vs Current")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Panel 2: KS separation chart
    if decile_tbl is not None and len(decile_tbl):
        axes[1].plot(decile_tbl["decile"], decile_tbl["cum_bad_pct"],
                     "o-", color="coral",     lw=2, label="Cum Bads %")
        axes[1].plot(decile_tbl["decile"], decile_tbl["cum_good_pct"],
                     "s-", color="steelblue", lw=2, label="Cum Goods %")
        axes[1].fill_between(decile_tbl["decile"],
                             decile_tbl["cum_bad_pct"],
                             decile_tbl["cum_good_pct"],
                             alpha=0.1, color="purple")
        axes[1].set_xlabel("Decile")
        axes[1].set_ylabel("Cumulative %")
        axes[1].set_title("KS Separation Chart")
        axes[1].legend()
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    chart_path = f"{out_dir}/monitoring_chart_{label}.png"
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.close()
    return chart_path


# ── MAIN: Score both datasets ─────────────────────────────────
_num_features = [f for f in features
                 if f in baseline_df.columns and f in current_df.columns]

X_base = baseline_df.reindex(columns=features, fill_value=0).values
X_curr = current_df.reindex(columns=features,  fill_value=0).values
p_base = model.predict_proba(X_base)[:, 1]
p_curr = model.predict_proba(X_curr)[:, 1]

# ── Score PSI ─────────────────────────────────────────────────
score_psi_val = _calculate_psi(p_base, p_curr)
score_psi_st  = _psi_status(score_psi_val)

# ── KS statistic + decile table ───────────────────────────────
ks_val = decile_tbl = None
if TARGET_COL in current_df.columns:
    y_curr    = current_df[TARGET_COL].values
    ks_val    = _calculate_ks(y_curr, p_curr)
    decile_tbl = _ks_decile_table(y_curr, p_curr)
ks_val = ks_val or 0.0

# ── Feature drift report ──────────────────────────────────────
drift_report = _run_drift_report(baseline_df, current_df, _num_features)
n_alerts = int(drift_report["alert"].sum()) if "alert" in drift_report.columns else 0

# ── Per-feature PSI table ─────────────────────────────────────
feat_psi_df = _psi_by_feature(baseline_df, current_df, _num_features)

# ── Default rate ──────────────────────────────────────────────
default_rate = float(current_df[TARGET_COL].mean() * 100) \
    if TARGET_COL in current_df.columns else None

# ── Action decision ───────────────────────────────────────────
action = (
    "RETRAIN"   if score_psi_val > 0.20 else
    "MONITOR"   if score_psi_val > 0.10 else
    "NO_ACTION"
)

# ── Save chart ────────────────────────────────────────────────
chart_path = _save_monitoring_chart(p_base, p_curr, decile_tbl, OUT_DIR, RUN_LABEL)

# ── Build report dict ─────────────────────────────────────────
monitoring_report = {
    "run_date":          datetime.now().isoformat(),
    "run_label":         RUN_LABEL,
    "baseline_rows":     len(baseline_df),
    "current_rows":      len(current_df),
    "score_psi":         round(score_psi_val, 4),
    "score_psi_status":  score_psi_st,
    "ks":                round(ks_val, 4),
    "ks_status":         _ks_status(ks_val),
    "default_rate_pct":  round(default_rate, 2) if default_rate is not None else None,
    "drift_alerts":      n_alerts,
    "features_checked":  len(_num_features),
    "action_required":   action,
    "top_drifting":      feat_psi_df.head(10).to_dict(orient="records"),
    "decile_table":      decile_tbl.to_dict(orient="records") if decile_tbl is not None else [],
}

# ── Save JSON reports ─────────────────────────────────────────
_report_path = f"{OUT_DIR}/monitoring_report_{RUN_LABEL}.json"
with open(_report_path, "w") as _f:
    json.dump(monitoring_report, _f, indent=2)
with open(f"{OUT_DIR}/monitoring_report_latest.json", "w") as _f:
    json.dump(monitoring_report, _f, indent=2)

# ── Save drift CSV ────────────────────────────────────────────
if len(drift_report):
    _drift_path = f"{OUT_DIR}/drift_report_{RUN_LABEL}.csv"
    drift_report.to_csv(_drift_path, index=False)

# ── OUTPUT ────────────────────────────────────────────────────
print(f"\n  Score PSI    : {score_psi_val:.4f}  →  {score_psi_st}")
print(f"  KS           : {ks_val:.4f}  →  {_ks_status(ks_val)}")
print(f"  Default rate : {default_rate:.2f}%" if default_rate is not None else "  Default rate : N/A")
print(f"  Drift alerts : {n_alerts} of {len(_num_features)} features checked")
print(f"  Action       : {action}")

print(f"\n  Saved:")
print(f"    {_report_path}")
print(f"    {chart_path}")
if len(drift_report):
    print(f"    {_drift_path}")

# ── KS Decile Table ───────────────────────────────────────────
if decile_tbl is not None:
    print(f"\n  KS Decile Table:")
    print("-" * 60)
    print(decile_tbl.to_string(index=False))
    print(f"  Max KS = {decile_tbl['ks_pct'].max():.2f}% at decile "
          f"{decile_tbl.loc[decile_tbl['ks_pct'].idxmax(), 'decile']}")

# ── Top drifting features ─────────────────────────────────────
print(f"\n  Top 10 Drifting Features (by PSI):")
print("-" * 60)
_drift_show = [c for c in ["feature", "psi", "status", "ks_stat", "ks_pval", "alert"]
               if c in feat_psi_df.columns]
print(feat_psi_df.head(10)[_drift_show].to_string(index=False))

print("\n" + "=" * 60)


  model loaded from disk  : GradientBoostingClassifier
  features loaded         : 86 features

  Score PSI    : 0.0000  →  STABLE
  KS           : 0.8665  →  EXCELLENT
  Default rate : 51.67%
  Drift alerts : 0 of 86 features checked
  Action       : NO_ACTION

  Saved:
    models/monitoring_report_202603.json
    models/monitoring_chart_202603.png
    models/drift_report_202603.csv

  KS Decile Table:
------------------------------------------------------------
 decile  n  bads  goods  bad_rate_pct  cum_bad_pct  cum_good_pct  ks_pct
      1 30    30      0        100.00        19.35          0.00   19.35
      2 30    29      1         96.67        38.06          0.69   37.37
      3 30    28      2         93.33        56.13          2.07   54.06
      4 30    29      1         96.67        74.84          2.76   72.08
      5 30    24      6         80.00        90.32          6.90   83.43
      6 30     9     21         30.00        96.13         21.38   74.75
      7 30     3     